# Experimento 9 - Drop Insulin, imputação por mediana e StandardScaler

## Dataset de treino

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

NORMALIZE_COLS = ['Glucose', 'BloodPressure', 'SkinThickness', 'BMI', 'DiabetesPedigreeFunction']

df = pd.read_csv("../../diabetes_dataset.csv")
df.shape


(572, 9)

### 1. Dropando `Insulin`

65% dos registros não tinham esse valor e, ao tentar imputar por mediana, criava um pico artificial enorme na coluna, então ela sai do dataset.

In [2]:
df = df.drop(columns=['Insulin'])
df.columns.tolist()


['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age',
 'Outcome']

### 2. Imputando NaN com a mediana

Guardamos as medianas calculadas aqui — elas também vão ser usadas no
dataset de previsão, pra não usar nenhuma estatística fora do treino.

In [3]:
medianas = df.median(numeric_only=True)
df = df.fillna(medianas)
df.isna().sum()

# medianas

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

### 3. Normalizando com StandardScaler

Só em `Glucose`, `BloodPressure`, `SkinThickness`, `BMI`,
`DiabetesPedigreeFunction`. `Pregnancies` e `Age` ficam brutas (são
contagens/anos, não medidas contínuas como as outras), e `Outcome` não é
tocado. O `scaler` é ajustado aqui e reaproveitado no dataset de previsão.

In [4]:
scaler = StandardScaler()
df[NORMALIZE_COLS] = scaler.fit_transform(df[NORMALIZE_COLS])
df.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,0.880461,-0.090370,0.665421,0.191861,0.499893,50,1
1,1,-1.221168,-0.592231,-0.075567,-0.858470,-0.321950,31,0
2,8,2.048032,-0.759518,0.047931,-1.353626,0.633890,32,1
3,0,0.513510,-2.766961,0.665421,1.617311,5.445843,33,1
4,5,-0.187033,0.076917,0.047931,-1.008518,-0.768604,30,0


### 4. Exportando

In [5]:
df.to_csv("diabetes-tratado-ex9.csv", index=False)
print("treino exportado com", len(df), "registros e colunas:", list(df.columns))


treino exportado com 572 registros e colunas: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


## Dataset de previsão (`diabetes_app.csv`)

Mesmas transformações do treino, mas reaproveitando as medianas e o
`scaler` já ajustados acima (sem `fit` de novo) — treino e previsão
precisam ficar na mesma escala pro KNN funcionar.

In [6]:
df_app = pd.read_csv("../../diabetes_app.csv")
df_app = df_app.drop(columns=['Insulin'])
df_app = df_app.fillna(medianas)
df_app[NORMALIZE_COLS] = scaler.transform(df_app[NORMALIZE_COLS])
df_app.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,BMI,DiabetesPedigreeFunction,Age
0,3,0.246636,-0.759518,-0.075567,-0.888480,-0.715006,28
1,3,2.314906,-0.424944,-1.804540,-0.213267,-0.476790,34
2,15,0.480151,-0.257657,0.294927,0.717027,-0.911534,43
3,1,-0.420547,-1.094092,-2.669026,-1.038527,1.452755,21
4,8,0.980538,0.411491,0.294927,1.587302,0.169369,36


In [7]:
df_app.to_csv("diabetes_app-tratado-ex9.csv", index=False)
print("app exportado com", len(df_app), "registros e colunas:", list(df_app.columns))


app exportado com 196 registros e colunas: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'BMI', 'DiabetesPedigreeFunction', 'Age']


# Resultado

O servidor de previsão retornou:
```
 - Resposta do servidor:
 {"status":"success","dev_key":"Omicron","accuracy":0.5867346938775511,"old_accuracy":0.64285714285714}
```